# DStreams and Structured Streaming — a simple comparison

Read one section at a time. Each introduces one difference, with tiny data and short code cells.
All processing examples use actual Spark streams. Saved outputs are included, so you can read without running anything.

1. How they work
2. Data structures and engines
3. Remembering state
4. Time windows
5. Checkpoints
6. Recovery after a restart
7. Late arrivals

Open **Start Simple Streaming Lesson.cmd**, then select **Run → Run All Cells**. Allow about 3–5 minutes.
Use **Kernel → Restart Kernel** before repeating the whole notebook.

## Setup — run once

The support file handles local Java setup, file delivery, waiting, and displaying results.
It does not contain the counting or window calculations. Those appear below.
`spark` works with DataFrames; `sc` is the SparkContext used by RDDs and DStreams.

In [1]:
from spark_lesson_support import *
spark, sc = start_spark()

Ready: Spark 3.5.7, two local worker threads.


## 1. How they work

**DStreams:** a stream is a sequence of RDD batches.
**Structured Streaming:** a stream is represented as a table receiving new rows.

Both use small batches by default here. Both can process partitions in parallel.
Neither means every record must be processed one by one on one machine.

### DStream: apply a function to each arriving record

We supply two test batches: `cat, dog`, followed by `cat`.

In [2]:
ssc = StreamingContext(sc, 5)  # A batch every 5 seconds.
first = sc.parallelize(["cat", "dog"])
second = sc.parallelize(["cat"])
ds_words = ssc.queueStream([first, second])
capital_words = ds_words.map(lambda word: word.upper())
show_batches(ssc, capital_words, 2)

RDD batch 1: ['CAT', 'DOG']
RDD batch 2: ['CAT']


`queueStream` supplies our two test RDDs. It is not a production source.
`map` applies the Python function to records in each batch.
`show_batches` starts the context, displays two output RDDs, then stops it.
Notice that batch 2 contains only its new `CAT`.

### Structured Streaming: describe a calculation on a growing input table

Each line in our input files is one word. `readStream` watches for new files.

In [3]:
word_input = new_folder("words")
ss_words = spark.readStream.text(word_input)
capital_table = ss_words.select(upper("value").alias("word"))
query = capital_table.writeStream.format("memory").queryName("capitals").start()

`select` chooses an output column. `upper` is a built-in Spark operation.
The input text column is named `value`; `alias` names the output column `word`.
`writeStream...start()` starts the calculation. Memory output is just for inspection.

In [4]:
add_text(word_input, "first.txt", "cat\ndog\n")
query.processAllAvailable()
spark.table("capitals").show()

+----+
|word|
+----+
| CAT|
| DOG|
+----+



In [5]:
add_text(word_input, "second.txt", "cat\n")
query.processAllAvailable()
spark.table("capitals").show()
query.stop()

+----+
|word|
+----+
| CAT|
| DOG|
| CAT|
+----+



The output table now contains three rows: `CAT`, `DOG`, `CAT` (row order is not guaranteed).
It retains rows already written to it. This uppercase calculation itself does not need running aggregation state.
`processAllAvailable` waits for current input to finish; it does not stop the query.

## 2. Data structures and engines

| DStreams | Structured Streaming |
|---|---|
| A sequence of RDDs | A streaming DataFrame with named, typed columns |
| Functions such as `map` run on records | Expressions such as `select`, `upper`, and `groupBy` describe a table calculation |
| RDD-based execution | Spark SQL engine; Catalyst plans supported expressions |

We can inspect the schema of the streaming DataFrame we just used:

In [6]:
ss_words.printSchema()
print("Is this a streaming DataFrame?", ss_words.isStreaming)

root
 |-- value: string (nullable = true)

Is this a streaming DataFrame? True


`value: string` means the input has a text column named `value`.

**Spark SQL is the engine's name, not a requirement to write SQL.** Python DataFrame operations also use it.
Our built-in `upper` expression is understood by that engine. Arbitrary Python functions are less visible to its optimizer.
This does not guarantee that every DataFrame calculation is faster.

A DStream batch can be converted to a DataFrame to use this engine for that batch.
That alone does not turn separate batches into one Structured Streaming query.

*The code below is how structured streaming write query*

In [ ]:
from pyspark.sql.functions import col, udf
from pyspark.sql.types import LongType

prices_input = folder("catalyst_stream_input")
items = (spark.readStream
         .schema("product STRING, price LONG")
         .csv(str(prices_input)))

# A built-in column expression is visible to Catalyst.
built_in = (items.withColumn("double_price", col("price") * 2)
            .filter(col("price") > 1).select("product", "double_price"))

# SQL over this streaming view also produces a streaming DataFrame.
items.createOrReplaceTempView("live_items")
sql_stream = spark.sql("SELECT product, price * 2 AS double_price FROM live_items WHERE price > 1")

# Spark knows this function's return type, but cannot inspect its Python body.
double_in_python = udf(lambda price: price * 2, LongType())
with_udf = (items.withColumn("double_price", double_in_python(col("price")))
            .filter(col("price") > 1).select("product", "double_price"))

query.explain() displays each streaming query's execution plan after processing data. Look for price * 2 in the built-in and SQL plans, and BatchEvalPython in the UDF plan. The built-in multiplication is visible to Catalyst. The UDF uses a separate Python evaluation step; Spark can still optimize surrounding operations, including the price filter.

spark.table(...).show() only displays a finite snapshot of the memory output; the calculation itself runs continuously through writeStream until we stop it. The memory output is for this demonstration. These plans are not a speed benchmark or a guarantee that one streaming model always performs better.

## 3. State — remembering previous input

Now count words **across batches**. Both models can do this.

| Input added | Running counts |
|---|---|
| `cat, dog` | cat = 1, dog = 1 |
| `cat` | cat = 2, dog = 1 |

### DStream: provide the state-update rule

The rule is: **new total = old total + the new values**.

In [7]:
ds_input = new_folder("ds_state_input")
ds_output = new_folder("ds_state_output")
ds_checkpoint = new_folder("ds_checkpoint")

def add_counts(new_values, old_total):
    return sum(new_values) + (old_total or 0)

`old_total or 0` starts a newly seen word at zero.
The next function defines the streaming calculation. We keep it so we can restore it later.

In [8]:
def make_counter():
    context = StreamingContext(sc, 10)
    context.checkpoint(ds_checkpoint)
    words = context.textFileStream(ds_input)
    pairs = words.map(lambda word: (word, 1))
    totals = pairs.updateStateByKey(add_counts)
    totals.checkpoint(10)
    save_batches(totals, ds_output)
    return context

`textFileStream` reads new text files. `map` turns `cat` into `('cat', 1)`.
`updateStateByKey` uses our update rule to maintain a total for each word.
The checkpoint calls enable saved progress/state; `10` requests an RDD checkpoint interval of 10 seconds.
`save_batches` only writes the resulting RDDs to files for inspection.

In [9]:
ssc = make_counter()
ssc.start()
feed_counts(ds_input, "cat\ndog\n", ds_output, {"cat": 1, "dog": 1})
feed_counts(ds_input, "cat\n", ds_output, {"cat": 2, "dog": 1})
ssc.stop(stopSparkContext=False, stopGraceFully=True)

[('cat', 1), ('dog', 1)]


[('cat', 2), ('dog', 1)]


`feed_counts` adds a file, waits for the expected counts, and prints the real output.
The last line stops this stream while keeping the shared SparkContext available.

### Structured Streaming: describe the result you want

Here the state is implied by `groupBy(...).count()`. We do not write an update function.

In [10]:
ss_input = new_folder("ss_state_input")
ss_output = new_folder("ss_state_output")
ss_checkpoint = new_folder("ss_checkpoint")
words = spark.readStream.text(ss_input)
totals = words.groupBy("value").count()

In [11]:
def start_counter():
    return (totals.writeStream
            .outputMode("complete")
            .option("checkpointLocation", ss_checkpoint)
            .foreachBatch(snapshot_writer(ss_output))
            .start())

Complete mode sends the whole current count table to the output on each update.
`snapshot_writer` saves that table to files; it does not calculate any counts.
We use file output here so the results can be inspected after a restart.

In [12]:
query = start_counter()
add_text(ss_input, "first.txt", "cat\ndog\n")
show_counts(query, ss_output)
add_text(ss_input, "second.txt", "cat\n")
show_counts(query, ss_output)
query.stop()

[('cat', 1), ('dog', 1)]


[('cat', 2), ('dog', 1)]


Both models produced `cat = 2, dog = 1`.
DStreams use our update rule. Structured Streaming recognizes the aggregation and manages its internal state.
Both distribute and update state automatically once configured.

## 4. Windows — which records belong together?

**Batch/arrival time:** when Spark receives the record.
**Event time:** the timestamp inside the record, describing when the event happened.

### DStream: a window over recent batch intervals

With 10-second batches, a 20-second window contains the latest two batch intervals.
It slides forward every 10 seconds.

In [13]:
ssc = StreamingContext(sc, 10)
a = sc.parallelize(["cat"])
b = sc.parallelize(["dog"])
c = sc.parallelize(["owl"])
animals = ssc.queueStream([a, b, c])
recent = animals.window(20, 10).countByValue()
show_batches(ssc, recent, 3)

RDD batch 1: [('cat', 1)]
RDD batch 2: [('cat', 1), ('dog', 1)]
RDD batch 3: [('dog', 1), ('owl', 1)]


Read the three outputs as: `cat`; then `cat + dog`; then `dog + owl`.
`cat` leaves when its batch interval is outside the window.
This standard DStream window does not inspect timestamps inside records.

### Structured Streaming: group by an event-time window

Both events below happened between 12:00 and 12:10, so they belong to the same window.
They arrive in separate files, with the 12:06 event arriving after the 12:08 event.

In [14]:
event_input = new_folder("event_windows")
events = spark.readStream.schema("event_time TIMESTAMP").json(event_input)
window_counts = events.groupBy(window("event_time", "10 minutes")).count()
query = window_counts.writeStream.format("memory").queryName("event_counts").outputMode("complete").start()

In [15]:
add_event(event_input, "first.json", "2026-01-01T12:08:00")
show_windows(spark, query, "event_counts")
add_event(event_input, "second.json", "2026-01-01T12:06:00")
show_windows(spark, query, "event_counts")
query.stop()

+-------------------+-------------------+-----+
|start              |end                |count|
+-------------------+-------------------+-----+
|2026-01-01 12:00:00|2026-01-01 12:10:00|1    |
+-------------------+-------------------+-----+



+-------------------+-------------------+-----+
|start              |end                |count|
+-------------------+-------------------+-----+
|2026-01-01 12:00:00|2026-01-01 12:10:00|2    |
+-------------------+-------------------+-----+



The 12:00–12:10 window changes from count 1 to count 2.
`window` groups by the event timestamp, regardless of arrival order.
This example has no watermark, so it has not set a limit for discarding old window state.

## 5. Checkpoints — save enough information to continue

**State** is working memory, such as `cat = 2`.
A **checkpoint** stores recovery information outside that working memory.

We already enabled checkpoints in section 3:

| DStreams | Structured Streaming |
|---|---|
| `context.checkpoint(ds_checkpoint)` | `.option("checkpointLocation", ss_checkpoint)` |
| Saves computation metadata and state RDD checkpoints | Saves input progress, managed state, and query completion information |

The source must still make input available for replay. A checkpoint is not automatically a backup of every input record.
Both examples use retained files. A plain socket would not offer the same replay behavior.

## 6. Recovery — stop, then continue from saved progress

Both counters stopped with `cat = 2, dog = 1`.
Add `dog, owl` after restarting. The correct result is **cat = 2, dog = 2, owl = 1**.

- If state were lost, `cat` would disappear.
- If all old input were counted again on top of restored state, `cat` would incorrectly increase.

### DStream: restore the saved StreamingContext

In [16]:
ssc = StreamingContext.getOrCreate(ds_checkpoint, make_counter)
ssc.start()
feed_counts(ds_input, "dog\nowl\n", ds_output, {"cat": 2, "dog": 2, "owl": 1})
ssc.stop(stopSparkContext=False, stopGraceFully=True)

[('cat', 2), ('dog', 2), ('owl', 1)]


`getOrCreate` loads the saved context if a checkpoint exists.
Otherwise it calls `make_counter` to define a new calculation.

### Structured Streaming: start the same query with the same checkpoint

In [17]:
query = start_counter()
add_text(ss_input, "third.txt", "dog\nowl\n")
show_counts(query, ss_output)
query.stop()

[('cat', 2), ('dog', 2), ('owl', 1)]


The function reuses the same `checkpointLocation`, so Spark restores the counts and input progress.
Both models recovered the expected result. This demonstrates an orderly stop/restart, not a sudden hardware failure.

**Recovery does not make arbitrary external writes happen exactly once.**
Sources must support replay, and outputs must handle retries correctly. Our snapshot writer replaces the output for a batch ID instead of appending another copy.

## 7. Late arrivals — an old event arrives now

An event can happen at 12:06 but arrive after an event from 12:08.

**DStreams:** a standard `.window(...)` places records by their batch interval.
It does not automatically move an arriving record back into its event-time window. Custom event-time handling requires additional application logic.

**Structured Streaming:** event-time windows can use a **watermark** to bound how long old state is kept.
Here we allow a 5-minute delay. The watermark advances with observed event timestamps, not simply with the computer's clock.

In [18]:
late_input = new_folder("late_events")
events = spark.readStream.schema("event_time TIMESTAMP").json(late_input)
final_counts = (events.withWatermark("event_time", "5 minutes")
               .groupBy(window("event_time", "10 minutes")).count())
query = final_counts.writeStream.format("memory").queryName("final_windows").outputMode("append").start()

`withWatermark` declares the late-data policy before the aggregation.
Append mode publishes a window's count only once Spark considers that window final.

First observe 12:08, then receive an out-of-order 12:06 event. Both are kept in the 12:00–12:10 window.

In [19]:
add_event(late_input, "first.json", "2026-01-01T12:08:00")
query.processAllAvailable()
add_event(late_input, "second.json", "2026-01-01T12:06:00")
show_windows(spark, query, "final_windows")

+-----+---+-----+
|start|end|count|
+-----+---+-----+
+-----+---+-----+



The output is empty **because the window is not final yet**, not because the events were discarded.

Now observe 12:20. With a 5-minute delay, the watermark can advance to 12:15.
That passes the old window's 12:10 end, allowing Spark to finalize it and remove its state.

In [20]:
add_event(late_input, "future.json", "2026-01-01T12:20:00")
show_windows(spark, query, "final_windows")

+-------------------+-------------------+-----+
|start              |end                |count|
+-------------------+-------------------+-----+
|2026-01-01 12:00:00|2026-01-01 12:10:00|2    |
+-------------------+-------------------+-----+



The finalized 12:00–12:10 window has **count 2**, proving the out-of-order 12:06 event was included.
The 12:20 event belongs to a newer window, which is not final yet.

Finally, receive a 12:04 event after the old window has been finalized. It is now too late for that window.

In [21]:
add_event(late_input, "too_late.json", "2026-01-01T12:04:00")
show_windows(spark, query, "final_windows")
show_late_rows(query)
query.stop()

+-------------------+-------------------+-----+
|start              |end                |count|
+-------------------+-------------------+-----+
|2026-01-01 12:00:00|2026-01-01 12:10:00|2    |
+-------------------+-------------------+-----+

Rows dropped as too late: 1


The old window stays at **count 2**, and the query reports **1 row dropped as too late**.
The 5-minute setting is not a promise that every older record is immediately dropped; watermark advancement and the query's window rules matter.

| Concept | DStreams | Structured Streaming |
|---|---|---|
| Input model | Sequence of RDDs | Streaming DataFrame |
| Built-in calculation engine | RDD operations | Spark SQL / Catalyst |
| Running state | Stateful operators plus an update rule | Managed state for the declared aggregation |
| Standard windows | Batch intervals | Built-in event-time windows |
| Recovery | Restore checkpointed context and state | Restart compatible query with the same checkpoint |
| Late events | Additional logic for event-time handling | Event-time windows and watermarks |

Both support state and recovery. Structured Streaming integrates table calculations, state, event time, and progress tracking in one query model.
Details: [DStream guide](https://spark.apache.org/docs/3.5.7/streaming-programming-guide.html) and [Structured Streaming guide](https://spark.apache.org/docs/3.5.7/structured-streaming-programming-guide.html).

In [22]:
spark.stop()
print("Finished.")

Finished.
